# CSCI 347 Final Project

Bailey Binando, Caitlin Hermanson, Lilly Pates

In [1]:
# load libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

In [2]:
# load data
df = pd.read_csv("data/master_full_rolling_5day_features.csv")
df.head() # glimpse

FileNotFoundError: [Errno 2] No such file or directory: 'data/master_full_rolling_5day_features.csv'

## Feature Selection

In [ ]:
# Correlation to impact for feature selection
# keep only numeric columns
num_df = df.select_dtypes(include = [np.number]).copy()

# drop rows with missing values for stable correlations
num_df = num_df.dropna()

# Spearman correlation with impact
corr_to_impact = (
    num_df.corr(method="spearman")["impact_proxy"].sort_values(ascending = False)
)

print(corr_to_impact)

In [ ]:
# Variation correlation to each other 
num_df = df.select_dtypes(include=[np.number]).dropna()
corr = num_df.corr(method="spearman")

upper = corr.abs().where(
    np.triu(np.ones(corr.shape), k=1).astype(bool)
)

high_corr = upper.stack().reset_index()
high_corr.columns = ["variable1", "variable2", "corr"]

high_corr = high_corr[high_corr["corr"] > 0.85]

# sort
high_corr = high_corr.sort_values("corr", ascending=False)

print(high_corr.to_string(index=False))

In [ ]:
# create month variable for future handling of missing values
df["month"] = pd.to_datetime(df["center_date"]).dt.month

# remove label/redundant analysis cols.
drop_cols = [
    "window_id", "center_date", "window_start", "window_end",
    "is_flood_event", "n_flood_days_in_window", "days_to_peak_flow",
    "days_to_nearest_flood_in_window", 'flow_raw_max',
    "impact_proxy", "impact_proxy_log", "rp_for_impact", 'rp_approx_max', 'rp_approx_mean',
    "window_n_days", "flow_raw_mean", "flow_pct_mean", "storm_wind_mean",
    "WS2M_mean", "RH2M_mean", 'api_k_0_70_max', 'api_k_0_70_mean', 'api_k_0_80_max',
    'api_k_0_80_mean', 'api_k_0_85_max', 'api_k_0_85_mean',
    "api_k_0_95_max", "api_k_0_95_mean", 'rain_1day_max', 'rain_1day_mean', 
    'rain_2day_max', 'rain_2day_mean', 'rain_4day_mean', 'rain_5day_max', 'rain_5day_mean', 'rain_7day_max',
    'rain_7day_mean', 'rain_14day_max', 'rain_14day_mean', 'tp_6h_intensity_mm_hr_mean',
    'tp_12h_intensity_mm_hr_max', 'tp_12h_intensity_mm_hr_mean', 'storm_max_wind_kt_max',
    'storm_max_wind_kt_mean', 'storm_within_300km', 'storm_within_500km', 'storm_wind_max',
    'max_exceedance_90', 'rain_4day_max', 'api_k_0_90_mean', 'rain_3day_mean'
]

feature_selection_final = df.drop(columns = drop_cols)

feature_selection_final.head()

## Preprocessing

In [ ]:
# find NAs in dataset
feature_selection_final.isna().sum()


In [ ]:
# FILL MISSING VALUES
D_raw = feature_selection_final.copy()

# fill missing meteoroligical values with monthly median
# original data from NASA starts in 1981
# monthly median to maintain seasonality
met_cols = [
    "RH2M_max", "WS2M_max"
]
for col in met_cols:
    D_raw[col] = D_raw[col].fillna(D_raw.groupby("month")[col].transform("median"))

# remove month col. (no longer needed)
D_raw = D_raw.drop(columns = "month")

# fill missing values in storm distance
# if there is no storm (hurricane/tropical cyclone) in the Atalantic coded as NA
# set NA distances to max distance + buffer
max_distance = D_raw["storm_min_distance_km"].max() # 13616.956 km
buffer = max_distance + 1000
D_raw["storm_min_distance_km"] = D_raw["storm_min_distance_km"].fillna(buffer) # fill missing
D_raw["no_storm"] = (D_raw["storm_min_distance_km"] > max_distance).astype(int)

# check
D_raw.isna().sum()

In [ ]:
# TRANSFORMATIONS

# view distributions
D_raw.hist(figsize = (14, 10), bins = 40)
plt.show() # skewed: rain_3day_max, tp_6h_mm_hr_max, api_k_0_90_max, area_above_90

D = D_raw.copy()

D["rain_3day_max"] = np.log1p(D["rain_3day_max"])
D["tp_6h_intensity_mm_hr_max"] = np.log1p(D["tp_6h_intensity_mm_hr_max"])
D["api_k_0_90_max"] = np.log1p(D["api_k_0_90_max"])
D["area_above_90"] = np.log1p(D["area_above_90"])
D = D.drop(columns = "no_storm") # drop labeling variable
D.head()

## Analysis

### PCA

In [ ]:
# standardize
scaler = StandardScaler()   
D_scaled = scaler.fit_transform(D)

# pca on normalized
pca_D_scaled = PCA() 
pca_transformed_D_scaled = pca_D_scaled.fit_transform(D_scaled)

# store component variance
explained_var = pca_D_scaled.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

In [ ]:
# initialize
total_var = 0
count = 0

# iterate through components till 90% of variance is reached
while total_var < 0.90:
    total_var += explained_var[count]
    count += 1

print(f"In order to capture at least 90% of the variance, {count} principal components are needed.")
    

In [ ]:
var_components_1_2 = explained_var[0] + explained_var[1]
print(f"The fraction of variance captured by PC1 and PC2 is {round(var_components_1_2, 4)}.")

In [ ]:
print(explained_var)

In [ ]:
# PLOT CAPTURED VARIANCE FIGURES

# X axis: number of components (1 to n)
components = np.arange(1, len(cumulative_var) + 1)

# Plot
plt.plot(components, cumulative_var, marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title("PCA of Flood-Related Features", fontsize = 10)
plt.suptitle('Cumulative Explained Variance by Principal Components')
plt.grid() # add gridlines

# Plot
plt.show()
plt.plot(components, explained_var, marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Explained Variance')
plt.suptitle('Explained Variance by Principal Component')
plt.title("PCA of Flood-Related Features", fontsize = 10)
plt.grid() # add gridlines

In [ ]:
# PLOT PCA COMPONENTS
plot_df = D.copy()

# get label cols.
plot_df["no_storm"] = D_raw["no_storm"].values
plot_df["is_flood_event"] = df["is_flood_event"]
plot_df["impact_proxy"] = df["impact_proxy"]
sizes = (plot_df["impact_proxy"] / plot_df["impact_proxy"].max()) * 100 # normalize for extreme impacts

flood = plot_df["is_flood_event"] == 1 # color by flood/non-flood event
no_flood = plot_df["is_flood_event"] == 0

# PC1 vs. PC2
plt.scatter(pca_transformed_D_scaled[flood, 0], pca_transformed_D_scaled[flood, 1], alpha = 0.3, s = sizes[flood], label = "Flood")
plt.scatter(pca_transformed_D_scaled[no_flood, 0], pca_transformed_D_scaled[no_flood, 1], alpha = 0.05, s = sizes[no_flood], label = "No Flood")
plt.legend()
plt.suptitle('PC1 vs. PC2')
plt.title("Colored by Flood Event, Sized by Impact", fontsize = 10)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.savefig("figure_pc1_2.png", dpi = 300, bbox_inches = "tight") # save as png

# PC1 vs. PC3
plt.show()
plt.scatter(pca_transformed_D_scaled[flood, 0], pca_transformed_D_scaled[flood, 2], alpha = 0.3, s = sizes[flood], label = "Flood")
plt.scatter(pca_transformed_D_scaled[no_flood, 0], pca_transformed_D_scaled[no_flood, 2], alpha = 0.05, s = sizes[no_flood], label = "No Flood")
plt.legend()
plt.suptitle('PC1 vs. PC3')
plt.title("Colored by Flood Event, Sized by Impact", fontsize = 10)
plt.xlabel("PC1")
plt.ylabel("PC3")
plt.savefig("figure_pc1_3.png", dpi = 300, bbox_inches = "tight") # save as png

# PC2 vs. PC3
plt.show()
plt.scatter(pca_transformed_D_scaled[flood, 1], pca_transformed_D_scaled[flood, 2], alpha = 0.3, s = sizes[flood], label = "Flood")
plt.scatter(pca_transformed_D_scaled[no_flood, 1], pca_transformed_D_scaled[no_flood, 2], alpha = 0.05, s = sizes[no_flood], label = "No Flood")
plt.legend()
plt.suptitle('PC2 vs. PC3')
plt.title("Colored by Flood Event, Sized by Impact", fontsize = 10)
plt.xlabel("PC2")
plt.ylabel("PC3")

plt.savefig("figure_pc2_3.png", dpi=300, bbox_inches="tight") # save as png

In [ ]:
# PLOT PCA COMPONENTS
# Look at relationship w/ storm

# PC1 vs. PC2
plt.show()
no_storm = plot_df["no_storm"] == 1 # color by storm/no-storm presence
storm = plot_df["no_storm"] == 0
plt.scatter(pca_transformed_D_scaled[no_storm, 0], pca_transformed_D_scaled[no_storm, 1], alpha = 0.05, label = "No Storm")
plt.scatter(pca_transformed_D_scaled[storm, 0], pca_transformed_D_scaled[storm, 1], alpha = 0.05, label = "Storm")
plt.legend()
plt.suptitle('PCA-Transformed Normalized Flood Data Using First Two Dimensions')
plt.title("Colored by Storm Presence", fontsize = 10)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.savefig("figure_pc1_2_storm.png", dpi = 300, bbox_inches = "tight") # save as png

# PC1 vs. PC3
plt.show()
no_storm = plot_df["no_storm"] == 1 # color by storm/no-storm presence
storm = plot_df["no_storm"] == 0
plt.scatter(pca_transformed_D_scaled[no_storm, 0], pca_transformed_D_scaled[no_storm, 2], alpha = 0.05, label = "No Storm")
plt.scatter(pca_transformed_D_scaled[storm, 0], pca_transformed_D_scaled[storm, 2], alpha = 0.05, label = "Storm")
plt.legend()
plt.suptitle('PCA-Transformed Normalized Flood Data Using First Two Dimensions')
plt.title("Colored by Storm Presence", fontsize = 10)
plt.xlabel("PC1")
plt.ylabel("PC3")
plt.savefig("figure_pc1_3_storm.png", dpi = 300, bbox_inches = "tight") # save as png

# PC2 vs. PC3
plt.show()
no_storm = plot_df["no_storm"] == 1 # color by storm/no-storm presence
storm = plot_df["no_storm"] == 0
plt.scatter(pca_transformed_D_scaled[no_storm, 1], pca_transformed_D_scaled[no_storm, 2], alpha = 0.05, label = "No Storm")
plt.scatter(pca_transformed_D_scaled[storm, 1], pca_transformed_D_scaled[storm, 2], alpha = 0.05, label = "Storm")
plt.legend()
plt.suptitle('PCA-Transformed Normalized Flood Data Using First Two Dimensions')
plt.title("Colored by Storm Presence", fontsize = 10)
plt.xlabel("PC1")
plt.ylabel("PC3")
plt.savefig("figure_pc2_3_storm.png", dpi = 300, bbox_inches = "tight") # save as png

In [ ]:
# VIEW PCA COMPONENTS
# store pca components & variable weights
components_pca = pd.DataFrame(pca_D_scaled.components_.T, index = D.columns)

# look at var. weights in each pc
pc1 = components_pca[0]
print("PC1")
print(pc1)
print()

print("PC2")
pc2 = components_pca[1]
print(pc2)
print()

print("PC3")
pc3 = components_pca[2]
print(pc3)
print()


## Clustering

In [ ]:
# Objective function value for each of the k value plot
# use first 3 PCs (~80% variance)
D_cluster = pca_transformed_D_scaled[:, :3] # PC1-5 minimal clustering benefit with added complexity
k_values = range(1, 10)
k_obj_pca = []

# iterate through each k-val
for k in k_values: 
    kmeans = KMeans(n_clusters = k, init = 'k-means++', n_init = 10) # run k-means algorithm
    kmeans.fit(D_cluster)
    k_obj_pca.append(kmeans.inertia_) # store
        

# plot elbow curve
plt.figure()
plt.plot(k_values, k_obj_pca, marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Objective Function')
plt.title('Elbow Plot: K-Means on PCA-Reduced Data')
plt.grid()
plt.axvline(x = 3, linestyle = '--', linewidth = 1, color = "red") # add verticle line when k = 3
plt.savefig("figure_k_means_elbow.png", dpi = 300, bbox_inches = "tight") # save as png
plt.show() # elbow when k = 3

In [ ]:
# Run k-means with k = 3
kmeans = KMeans(n_clusters = 3, init = 'k-means++', max_iter = 300, random_state = 0) # run k-means algorithm
pred_cluster_labels = kmeans.fit_predict(D_cluster) # index of the cluster each sample belongs to
kmeans.cluster_centers_  # coordinates of cluster centers

# scatter plot of clusters & centroids
# PC1 vs. PC2
plt.scatter(D_cluster[:,0], D_cluster[:, 1], c=pred_cluster_labels, alpha = 0.05)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], c = 'red')
plt.title("PC1 vs. PC2")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.savefig("figure_k_means_clustering_pc1_2.png", dpi = 300, bbox_inches = "tight") # save as png

# PC1 vs. PC3
plt.show()
plt.scatter(D_cluster[:,0], D_cluster[:, 2], c=pred_cluster_labels, alpha = 0.05)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 2], c = 'red')
plt.title("PC1 vs. PC3")
plt.xlabel("PC1")
plt.ylabel("PC3")
plt.savefig("figure_k_means_clustering_pc1_3.png", dpi = 300, bbox_inches = "tight") # save as png

# PC2 vs. PC3
plt.show()
plt.scatter(D_cluster[:,1], D_cluster[:, 2], c=pred_cluster_labels, alpha = 0.05)
plt.scatter(kmeans.cluster_centers_[:, 1], kmeans.cluster_centers_[:, 2], c = 'red')
plt.title("PC2 vs. PC3")
plt.xlabel("PC2")
plt.ylabel("PC3")
plt.savefig("figure_k_means_clustering_pc2_3.png", dpi = 300, bbox_inches = "tight") # save as png

# calc mean impact and flooding within each cluster
plot_df["cluster"] = pred_cluster_labels
plot_df.groupby("cluster")[["impact_proxy", "is_flood_event"]].mean()


In [ ]:
# Calc Silhoutte Score
# k = 3
sil_score = silhouette_score(D_cluster, pred_cluster_labels)
print("Silhoutte Score: k = 3")
print(sil_score)
print()

# compare to other cluster sizes
# Run k-means with k = 2
kmeans2 = KMeans(n_clusters = 2, init = 'k-means++', max_iter = 300, random_state = 0) # run k-means algorithm
pred_cluster_labels2 = kmeans2.fit_predict(D_cluster) # index of the cluster each sample belongs to
kmeans2.cluster_centers_  # coordinates of cluster centers
sil_score2 = silhouette_score(D_cluster, pred_cluster_labels2)
print("Silhoutte Score: k = 2")
print(sil_score2)
print()

# Run k-means with k = 4
kmeans4 = KMeans(n_clusters = 4, init = 'k-means++', max_iter = 300, random_state = 0) # run k-means algorithm
pred_cluster_labels4 = kmeans4.fit_predict(D_cluster) # index of the cluster each sample belongs to
kmeans4.cluster_centers_  # coordinates of cluster centers
sil_score4 = silhouette_score(D_cluster, pred_cluster_labels4)
print("Silhoutte Score: k = 4")
print(sil_score4)
print()

In [ ]:
# DBSCAN
# run dbscan on pca transformed-normalized boston data
db = DBSCAN(eps = 0.5, min_samples = 5)
pred_labels = db.fit_predict(D_cluster)

# number of clusters in labels, ignoring noise if present
n_clusters_ = len(set(pred_labels)) - (1 if -1 in pred_labels else 0)
n_noise_ = list(pred_labels).count(-1)

print("Estimated number of clusters: %d" % n_clusters_) # 3
print("Estimated number of noise points: %d" % n_noise_) # 23

# plot data
plt.scatter(D_cluster[:,0], D_cluster[:,1], c = pred_labels, alpha = 0.5)
plt.title("DBSCAN Clustering on PCA-Transformed Normalized Data")
plt.xlabel("PC1")
plt.ylabel("PC2")

## Trigger Evaluation

In [ ]:
# add PCA scores and cluster labels to original dataset
analysis_df = df.copy()

analysis_df["PC1"] = pca_transformed_D_scaled[:, 0] # add PC1
analysis_df["PC2"] = pca_transformed_D_scaled[:, 1] # add PC2
analysis_df["PC3"] = pca_transformed_D_scaled[:, 2] # add PC3
analysis_df["cluster"] = pred_cluster_labels # add clusters

In [ ]:
# 2 triggers to test
# create binary trigger flags using rolling-window features

# Trigger 1: Flow & API (Compound/Balanced)
analysis_df["trigger1"] = ((analysis_df["flow_pct_max"] >= 0.96) & (analysis_df["api_k_0_95_max"].rank(pct=True) >= 0.90)).astype(int)

# Trigger 2: Flow or API (OR Trigger)
analysis_df["trigger2"] = ((analysis_df["flow_pct_max"] >= 0.975) | (analysis_df["api_k_0_90_max"].rank(pct=True) >= 0.90)).astype(int)

In [ ]:
# trigger types
trigger_cols = ["trigger1", "trigger2"]

# classify each row as TP/TN/FP/FN for each trigger type
for col in trigger_cols:
    case_col = col + "_case"

    analysis_df[case_col] = "TN" # assign everything as TN

    # reassign labels
    analysis_df.loc[(analysis_df[col] == 1) & (analysis_df["is_flood_event"] == 1), case_col] = "TP" 
    analysis_df.loc[(analysis_df[col] == 1) & (analysis_df["is_flood_event"] == 0), case_col] = "FP"
    analysis_df.loc[(analysis_df[col] == 0) & (analysis_df["is_flood_event"] == 1), case_col] = "FN"

# glimpse
analysis_df.head()


In [ ]:
# Summary of trigger performance
# Confusion Matrix + Precision + Recall + F1 Score

summary = [] 
for col in trigger_cols: # iterate through each trigger type
    trigger = analysis_df[col]
    flood = analysis_df["is_flood_event"]

    # confusion matrix
    tp = ((trigger == 1) & (flood == 1)).sum() # true positive
    fp = ((trigger == 1) & (flood == 0)).sum() # false positive
    fn = ((trigger == 0) & (flood == 1)).sum() # false negative
    tn = ((trigger == 0) & (flood == 0)).sum() # true negative

    # calc performance metrics
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = (2 * precision * recall) / (precision + recall)

    # create summary table
    summary.append({"trigger": col,
                    "tp": tp,
                    "fp": fp,
                    "fn": fn,
                    "tn": tn,
                    "precision": precision,
                    "recall": recall,
                    "f1": f1})
summary_df = pd.DataFrame(summary)
print(summary_df)


In [ ]:
### Trigger Comparison (PCA)

In [ ]:
# Flow + API Trigger
case_col = "trigger1_case"

# normalize impact (prevent extreme vals.)
sizes_impact = (analysis_df["impact_proxy"] / analysis_df["impact_proxy"].max()) * 100

plt.figure()

for case in ["TN", "FP", "FN", "TP"]:
    subset = analysis_df[analysis_df[case_col] == case]

    plt.scatter(subset["PC1"], subset["PC2"], s = sizes_impact[subset.index], alpha = 0.15, label = case)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.suptitle("Predicted Flow AND API (k = 0.95)")
plt.title("Colored by Confusion Class, Sized by Impact", fontsize = 10)
plt.legend()
plt.grid()
plt.savefig("figure_pca_trig1_compariso .png", dpi = 300, bbox_inches = "tight") # save as png
plt.show()

# OR Trigger
case_col = "trigger2_case"

for case in ["TN", "FP", "FN", "TP"]:
    subset = analysis_df[analysis_df[case_col] == case]
    plt.scatter(subset["PC1"], subset["PC2"], s = sizes_impact[subset.index], alpha = 0.15, label = case)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.suptitle("Predicted Flow OR API (k = 0.90)")
plt.title("Colored by Confusion Class, Sized by Impact", fontsize = 10)
plt.legend()
plt.grid()
plt.savefig("figure_pca_trig2_comparison.png", dpi = 300, bbox_inches = "tight") # save as png
plt.show()



In [ ]:
case_col = "trigger1_case"

# only select trigger 1 FP & FN
subset = analysis_df[analysis_df[case_col].isin(["FN", "FP"])].copy()

# color by cluster
cluster_colors = {0: "blue", 1: "green", 2: "gray"}

# Plot errors by cluster
for cluster in subset["cluster"].unique(): # loop through the 3 clusters
    data = subset[subset["cluster"] == cluster] 

    # split clusters by error type
    fn = data[data[case_col] == "FN"]
    fp = data[data[case_col] == "FP"]

    # plot FN - triangle
    plt.scatter(fn["PC1"], fn["PC2"], c = cluster_colors[cluster], marker = "^", 
                alpha = 0.5, label = f"Cluster {cluster + 1} - FN") # account for 0 indexing
    # plot FP - X
    plt.scatter(fp["PC1"], fp["PC2"], c = cluster_colors[cluster], marker = "x", 
                alpha = 0.5, label = f"Cluster {cluster + 1} - FP")

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.suptitle("Trigger Errors by Cluster in PCA Flood Feature Space")
plt.title("Predicted Flow AND API (k = 0.95)", fontsize = 10)
plt.grid()
plt.legend()
plt.savefig("figure_pca_clustering_errors.png", dpi = 300, bbox_inches = "tight") # save as png
plt.show()


In [ ]:
# calc confusion matrix proportion in each cluster
trigger1_cluster = pd.crosstab(analysis_df["cluster"], analysis_df["trigger1_case"], normalize="index")
print(trigger1_cluster)


In [ ]:
# summary of trigger outcomes
analysis_df.groupby("trigger1_case")["impact_proxy"].describe()

In [ ]:
# boxplot of impact distribution
analysis_df.boxplot(column = "impact_proxy", by = "trigger1_case")

plt.suptitle("Impact Distribution by Trigger Outcome")
plt.title("Predicted Flow AND API (k = 0.95)", fontsize = 10)
plt.xlabel("Trigger Outcome")
plt.ylabel("Impact Proxy (m²)")
plt.savefig("figure_trigger1_impact.png", dpi = 300, bbox_inches = "tight") # save as png
plt.show()

In [ ]:
analysis_df.groupby(["cluster", "trigger1_case"])["impact_proxy"].mean()

In [ ]:
case_col = "trigger1_case"

fn_df = analysis_df[analysis_df[case_col] == "FN"]  # missed floods
fp_df = analysis_df[analysis_df[case_col] == "FP"]  # false alarms
tp_df = analysis_df[analysis_df[case_col] == "TP"]
tn_df = analysis_df[analysis_df[case_col] == "TN"]

In [ ]:
# Plot Trigger 1 missed floods
# Filter FN only
fn_df = analysis_df[analysis_df[case_col] == "FN"]

# Normalize sizes
sizes = (fn_df["impact_proxy"] / analysis_df["impact_proxy"].max()) * 100

# plot all points
plt.scatter(analysis_df["PC1"], analysis_df["PC2"], c = "lightgray", alpha = 0.01)
# FN points (highlighted + sized by impact)
plt.scatter(fn_df["PC1"], fn_df["PC2"], c = "red", s = sizes, alpha = 0.4, label = "FN (Missed Floods)")

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Missed Floods (FN) in PCA Space (Sized by Impact)")
plt.legend()
plt.grid()
plt.show()


In [ ]:
# Plot Trigger 1 false floods
# Normalize sizes
sizes = (fp_df["impact_proxy"] / analysis_df["impact_proxy"].max()) * 100

plt.figure()

# plot all points
plt.scatter(analysis_df["PC1"], analysis_df["PC2"], c = "lightgray", alpha = 0.01)
# FP points (highlighted + sized by impact)
plt.scatter(fp_df["PC1"], fp_df["PC2"], c = "green", s = sizes, alpha = 0.4, label = "FP (False Floods)")

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("False Floods (FP) in PCA Space (Sized by Impact)")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# create API percentile columns
analysis_df["api_k_0_90_pct"] = analysis_df["api_k_0_90_max"].rank(pct=True)
analysis_df["api_k_0_95_pct"] = analysis_df["api_k_0_95_max"].rank(pct=True)

# summarize confusion matrix for trigger1
analysis_df.groupby(case_col)[["flow_pct_max","api_k_0_95_pct", "PC1", "impact_proxy"]].mean()

In [ ]:
analysis_df.groupby(case_col)[["flow_pct_max","api_k_0_95_pct", "PC1", "impact_proxy"]].mean()